- **2. 生成 dataset.json**：编写 Python 脚本，将 `step_4_metadata.csv` 转换为 JSON。路径全部指向**原始的 `.nii.gz` 文件**（不再指向任何 Latent 缓存）。
    - 映射关系：`image` -> Sub；`condition` -> Pre；`label` -> Mask。

In [1]:
import pandas as pd
import json
import os
from sklearn.model_selection import train_test_split

# --- 配置区 ---
CSV_PATH = "/mnt/ext_drive/breast/step_4/step_4_metadata.csv"
SAVE_PATH = "breast_project/data/dataset.json"

def get_source_name(original_id):
    """
    从 Original_ID 中提取数据集来源 (DUKE, ISPY1, ISPY2, NACT)
    如果你的 ID 命名不完全是这样，可以微调这里的匹配逻辑
    """
    orig_id = str(original_id).upper()
    if 'DUKE' in orig_id: return 'DUKE'
    if 'ISPY1' in orig_id: return 'ISPY1'
    if 'ISPY2' in orig_id: return 'ISPY2'
    if 'NACT' in orig_id: return 'NACT'
    return 'OTHER' # 备用兜底

def generate_maisi_json():
    # 1. 加载数据
    df = pd.read_csv(CSV_PATH)
    
    # 2. 提取数据源，并创建一个用于分层的联合特征 (Source + Has_Tumor)
    df['Source'] = df['Original_ID'].apply(get_source_name)
    df['Stratify_Key'] = df['Source'] + "_Tumor_" + df['Has_Tumor'].astype(str)
    
    # --- [数据分布分析] ---
    print("\n" + "="*40)
    print("📊 你的全量数据分布 (按来源与有无肿瘤):")
    print("="*40)
    distribution = df.groupby(['Source', 'Has_Tumor']).size().unstack(fill_value=0)
    distribution['Total'] = distribution.sum(axis=1)
    print(distribution)
    print(f"\n总数据量: {len(df)}")
    print("="*40 + "\n")

    # 3. 分层抽样 (Stratified Split)
    # test_size=100 表示我们强制让验证集精确为 100 个样本
    # stratify=df['Stratify_Key'] 保证了这 100 个样本的比例与全集的比例绝对一致
    try:
        train_df, val_df = train_test_split(
            df, 
            test_size=100, 
            stratify=df['Stratify_Key'], 
            random_state=42 # 固定随机种子，保证每次运行划分结果一致
        )
    except ValueError as e:
        # 兜底：如果某种组合的数量少于 2 个，sklearn 会报错。
        # 此时退化为仅按 'Source' 分层
        print("⚠️ 警告: 某种组合数量过少，退化为仅按数据源分层...")
        train_df, val_df = train_test_split(
            df, test_size=100, stratify=df['Source'], random_state=42
        )

    # --- [验证集质量检查] ---
    print("🧪 抽取出的 100 个验证集分布情况:")
    val_dist = val_df.groupby(['Source', 'Has_Tumor']).size().unstack(fill_value=0)
    print(val_dist)
    print("="*40 + "\n")

    # 4. 构建 MAISI JSON 格式
    dataset = {
        "description": "Breast Subtraction Generation Dataset (Stratified)",
        "labels": {"0": "background", "1": "tumor"},
        "tensorImageSize": "3D",
        "training": [],
        "validation": []
    }

    # 转换 DataFrame 为 JSON 需要的字典格式
    def row_to_dict(row):
        return {
            "image": row['Path_Sub'],      # 目标图
            "condition": row['Path_Pre'],  # 条件图
            "label": row['Path_Mask'],     # 掩模
            "has_tumor": int(row['Has_Tumor']),
            "uuid": row['UUID']
        }

    dataset["training"] = train_df.apply(row_to_dict, axis=1).tolist()
    dataset["validation"] = val_df.apply(row_to_dict, axis=1).tolist()

    # 5. 保存
    # 确保保存目录存在
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    with open(SAVE_PATH, 'w') as f:
        json.dump(dataset, f, indent=4)
    
    print(f"✅ 成功！高质量平衡 JSON 已保存至: {SAVE_PATH}")
    print(f"📈 最终训练集: {len(dataset['training'])} | 验证集: {len(dataset['validation'])}")

if __name__ == "__main__":
    generate_maisi_json()


📊 你的全量数据分布 (按来源与有无肿瘤):
Has_Tumor    0    1  Total
Source                    
DUKE       283  289    572
ISPY1        4  171    175
ISPY2      153  979   1132
NACT         0   64     64

总数据量: 1943

🧪 抽取出的 100 个验证集分布情况:
Has_Tumor   0   1
Source           
DUKE       15  15
ISPY1       0   9
ISPY2       8  50
NACT        0   3

✅ 成功！高质量平衡 JSON 已保存至: breast_project/data/dataset.json
📈 最终训练集: 1843 | 验证集: 100


In [2]:
import pandas as pd
import json
import os

# --- 本地配置 ---
ORIGINAL_CSV = "/mnt/ext_drive/breast/step_4/step_4_metadata.csv" 
# 确保保存路径包含文件名
SAVE_JSON_FULL_PATH = "/mnt/ext_drive/breast/breast-sub-gen/breast_project/data/vae_dataset.json"

# --- 目标环境配置 (Colab) ---
# 即使在本地运行，JSON 里的路径也必须指向 Colab 的位置
COLAB_BASE_DIR = "/content/step_4"

def generate_vae_json_for_colab():
    if not os.path.exists(ORIGINAL_CSV):
        print(f"❌ 本地找不到 CSV 文件: {ORIGINAL_CSV}")
        return

    df = pd.read_csv(ORIGINAL_CSV)
    
    all_images = []
    
    for _, row in df.iterrows():
        # 提取原始文件名，并拼接成 Colab 上的绝对路径
        pre_filename = os.path.basename(row['Path_Pre'])
        sub_filename = os.path.basename(row['Path_Sub'])
        
        colab_pre_path = os.path.join(COLAB_BASE_DIR, pre_filename)
        colab_sub_path = os.path.join(COLAB_BASE_DIR, sub_filename)
        
        # 摊平数据：每个样本贡献一个 Pre 和一个 Sub
        all_images.append({"image": colab_pre_path, "class": "mri"})
        all_images.append({"image": colab_sub_path, "class": "mri"})

    # 验证集划分：依然保留 100 对 (200 个条目)
    train_list = all_images[:-200]
    val_list = all_images[-200:]

    vae_data = {
        "description": "Flattened Breast Pre and Sub for VAE Fine-tuning (Generated Locally for Colab)",
        "training": train_list,
        "validation": val_list
    }

    # 确保本地目标目录存在
    os.makedirs(os.path.dirname(SAVE_JSON_FULL_PATH), exist_ok=True)

    with open(SAVE_JSON_FULL_PATH, 'w') as f:
        json.dump(vae_data, f, indent=4)
    
    print(f"✅ 成功生成本地 JSON: {SAVE_JSON_FULL_PATH}")
    print(f"📈 包含条目: 训练 {len(train_list)} | 验证 {len(val_list)}")
    print(f"🚀 请将此文件上传至 Colab，并确保数据解压在 {COLAB_BASE_DIR}")

if __name__ == "__main__":
    generate_vae_json_for_colab()

✅ 成功生成本地 JSON: /mnt/ext_drive/breast/breast-sub-gen/breast_project/data/vae_dataset.json
📈 包含条目: 训练 3686 | 验证 200
🚀 请将此文件上传至 Colab，并确保数据解压在 /content/step_4
